# get similarity and coherence

In [1]:
import os
import re

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

In [2]:
research1_dir = os.getcwd()

# loading the word2vec model
word2vec_model = KeyedVectors.load_word2vec_format(os.getcwd() + '/../pretrained/GoogleNews-vectors-negative300.bin', binary=True)
# print(word2vec_model['king']) # 모델이 잘 로드되었는지 확인

## 데이터셋 준비

In [11]:
processed_data_dir = os.getcwd() + '/data/processed/' # data위치 지정

pilot_data = pd.read_csv(processed_data_dir + 'merged_data.csv')
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend21,friend22,friend23,friend24,friend25,friend26,friend27,friend28,friend29,friend30
0,5d53bffa147a7d00015aae5a,1,Door,Gate,Outside,Grass,Itchy,Rash,Chicken pox,Shingles,...,Burned down,Scary,Movies,Knocked up,Tinsletown,Christmas tree,Holiday,Decorate,Fun,Party
1,5f00ec86304f7322eb8dfa41,2,Unlock,Door,Enter,Dreams,Time and space,Possibilites,Infinite,Time,...,Transparent,Glass,Shattered,Repair,Meaningful,Bond,Glue,All together,Matters,Wonders
2,5de27ced22383629b807cc70,3,Hole,Ground,Hog,Pig,Pork,Sald,Dressing,Clothes,...,Sunshine,Wamrth,Blanket,Heavy,Body,Muscle,Protein,Shake,Dance,Jump


In [12]:
seed_words = ['key', 'money', 'friend']
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(pilot_data) # 58
n_dim_of_vector = 300

In [13]:
for seed_word in seed_words: ##### 차례로 seed_word에 key, money, friend가 저장됨
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # key1~30, money1~30, friend1~30

    for column in word_columns: ##### 차례로 column에 데이터의 컬럼 하나씩 저장됨. ex) key1
        # vector field 생성
        pilot_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # remove white space after the word such as 'stoma ' and ' stoma'
        pilot_data[column] = pilot_data[column].str.strip()
        # Remove special character
        pilot_data[column] = pilot_data[column].apply(lambda x: re.sub('[^\w\s]', '', str(x)) if isinstance(x, (str, np.ndarray)) else x)
        # Lower case로 통일
        pilot_data[column] = pilot_data[column].str.lower()
        
        ## exceptional cases
        # typo fix: wamrth
        pilot_data[column] = pilot_data[column].replace('wamrth', 'warmth')

        for i_subject in range(n_subject): ##### 차례로 피험자 숫자ID(0~57)
            try:
                response_word = pilot_data.iloc[i_subject][column]

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]

                    vec_word2vec = np.zeros((n_dim_of_vector, 0)) # 벡터를 저장할 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        
                        # reshape함수: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기

                    pilot_data[column + '_vec'][i_subject] = vec_word2vec

            except:
                pass

/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_5827/3495203254.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_data[column + '_vec'][i_subject] = vec_word2vec
/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_5827/3495203254.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_data[column + '_vec'][i_subject] = vec_word2vec
/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_5827/3495203254.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_

In [14]:
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend21_vec,friend22_vec,friend23_vec,friend24_vec,friend25_vec,friend26_vec,friend27_vec,friend28_vec,friend29_vec,friend30_vec
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,"[[0.306640625, 0.0245361328125], [-0.005767822...","[[0.171875], [-0.1240234375], [0.1748046875], ...","[[0.01361083984375], [0.138671875], [-0.163085...","[[0.10009765625, 0.1201171875], [-0.0094604492...","[[0.1328125], [0.055908203125], [-0.1982421875...","[[-0.1689453125, 0.484375], [0.03466796875, 0....","[[0.275390625], [0.1201171875], [-0.212890625]...","[[0.08642578125], [0.047607421875], [-0.117675...","[[0.0791015625], [-0.1201171875], [-0.09423828...","[[-0.09130859375], [-0.0869140625], [-0.012084..."
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time and space,possibilites,infinite,time,...,"[[-0.287109375], [0.0179443359375], [-0.205078...","[[-0.2236328125], [0.1240234375], [-0.09130859...","[[0.033203125], [0.1953125], [0.0260009765625]...","[[-0.03173828125], [0.392578125], [-0.10302734...","[[-0.1279296875], [-0.035400390625], [-0.08007...","[[0.1923828125], [-0.09619140625], [0.18164062...","[[0.056640625], [0.047607421875], [-0.02124023...","[[-0.0078125, -0.1083984375], [-0.027954101562...","[[0.0308837890625], [0.25390625], [-0.04711914...","[[0.1708984375], [0.11376953125], [0.050292968..."
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,sald,dressing,clothes,...,"[[0.00823974609375], [0.07373046875], [0.04321...","[[0.2490234375], [0.13671875], [0.025390625], ...","[[0.02783203125], [-0.062255859375], [0.222656...","[[0.2890625], [0.306640625], [-0.15234375], [0...","[[-0.01129150390625], [-0.020751953125], [0.38...","[[0.46875], [0.279296875], [-0.006988525390625...","[[-0.12109375], [0.00136566162109375], [-0.007...","[[-0.0830078125], [-0.11474609375], [-0.056640...","[[0.18359375], [-0.318359375], [0.205078125], ...","[[0.052001953125], [0.0361328125], [-0.1035156..."


# similarity - coherence

- 피험자가 응답한 모든 단어 ↔ `key` 간의 similarity 
- 피험자가 응답한 모든 단어 ↔ `money` 간의 similarity
- 피험자가 응답한 모든 단어 ↔ `friend` 간의 similarity \
\
\
-> coherence값 구함

In [15]:
target_words = ['key', 'money', 'friend'] # seed_words와 동일

# coherence 컬럼 미리 생성(빈 값)
for seed_word in seed_words:
    for target_word in target_words:
        column_name = f'coherence_{seed_word}_{target_word}'
        pilot_data = pilot_data.assign(**{column_name: None})

pilot_data.iloc[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend30_vec,coherence_key_key,coherence_key_money,coherence_key_friend,coherence_money_key,coherence_money_money,coherence_money_friend,coherence_friend_key,coherence_friend_money,coherence_friend_friend
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,"[[-0.09130859375], [-0.0869140625], [-0.012084...",None,None,None,None,None,None,None,None,None
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time and space,possibilites,infinite,time,...,"[[0.1708984375], [0.11376953125], [0.050292968...",None,None,None,None,None,None,None,None,None
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,sald,dressing,clothes,...,"[[0.052001953125], [0.0361328125], [-0.1035156...",None,None,None,None,None,None,None,None,None


In [16]:
for seed_word in seed_words: ##### seed단어 하나씩 차례로 돌면서
    coherences_key_per_sub = []
    coherences_money_per_sub = []
    coherences_friend_per_sub = []

    word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)]

    for i_subject in range(n_subject): #### 각 피험자마다
        each_seed_similarities_key = []  
        each_seed_similarities_money = []  
        each_seed_similarities_friend = []

        for column in word_columns: ##### 각각 seed1~30
            try:
                similarity_between_key = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'key')
                similarity_between_money = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'money')
                similarity_between_friend = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'friend')
                each_seed_similarities_key.append(similarity_between_key) # 30개
                each_seed_similarities_money.append(similarity_between_money) # 30개
                each_seed_similarities_friend.append(similarity_between_friend) # 30개
            except:
                continue

        # 해당 피험자가 하나의 seed에 답한 30개의 응답단어의 유사도의 coherence값
        # i. 유사도의 평균값
        key_coherence_of_seed_per_sub = np.mean(each_seed_similarities_key) 
        money_coherence_of_seed_per_sub = np.mean(each_seed_similarities_money) 
        friend_coherence_of_seed_per_sub = np.mean(each_seed_similarities_friend)

        # 해당 피험자에 대한, 그 seed단어 각각(3개)에 대한 coherence값
        # 피험자마자 9개 값( key,money,friend - key,money,friend 조합)
        coherences_key_per_sub.append(key_coherence_of_seed_per_sub)
        coherences_money_per_sub.append(money_coherence_of_seed_per_sub)
        coherences_friend_per_sub.append(friend_coherence_of_seed_per_sub)

        # Append the coherence values to the data frame for the current subject
        pilot_data.at[i_subject, f'coherence_{seed_word}_key'] = key_coherence_of_seed_per_sub
        pilot_data.at[i_subject, f'coherence_{seed_word}_money'] = money_coherence_of_seed_per_sub
        pilot_data.at[i_subject, f'coherence_{seed_word}_friend'] = friend_coherence_of_seed_per_sub



/Users/ihyeseung/.pyenv/versions/3.9.18/envs/research/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/ihyeseung/.pyenv/versions/3.9.18/envs/research/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [17]:
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend30_vec,coherence_key_key,coherence_key_money,coherence_key_friend,coherence_money_key,coherence_money_money,coherence_money_friend,coherence_friend_key,coherence_friend_money,coherence_friend_friend
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,"[[-0.09130859375], [-0.0869140625], [-0.012084...",0.041143,0.102066,0.075201,0.030857,0.145726,0.063208,0.039349,0.097582,0.148602
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time and space,possibilites,infinite,time,...,"[[0.1708984375], [0.11376953125], [0.050292968...",0.096018,0.140075,0.120077,0.053342,0.129493,0.117039,0.081238,0.10836,0.105842
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,sald,dressing,clothes,...,"[[0.052001953125], [0.0361328125], [-0.1035156...",0.077837,0.115665,0.17007,0.057519,0.099424,0.132149,0.049724,0.145488,0.082121


In [22]:
drop_columns = pilot_data.columns[92:182] # vector 컬럼들 드롭
pilot_data = pilot_data.drop(drop_columns, axis='columns')

# 단어 있는 버전
pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data_300_with_words.csv', index=None)

In [29]:
# 단어 없이 coherence만 있는 버전
drop_columns = pilot_data.columns[2:92] # vector 컬럼들 드롭
pilot_data = pilot_data.drop(drop_columns, axis='columns')

pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data_300.csv', index=None)